<a href="https://colab.research.google.com/github/ravishankar-cloud/machine-learning/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# Tried coding based on Comparative Study of Marathi Text Classification Using Monolingual and Multilingual Embeddings
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ==========================================
# 1. Configuration Constants
# ==========================================
MODEL_NAME = "google/muril-base-cased"
BATCH_SIZE = 16  # Efficient batch scale for real datasets in Colab
MAX_LEN = 64
EPOCHS = 3
NUM_CLASSES = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Execution Target Device: {DEVICE}")

# ==========================================
# 2. Dataset Custom Class
# ==========================================
class MarathiTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# ==========================================
# 3. Model Architecture (MuRIL + BiLSTM Setup)
# ==========================================
class MurilBiLSTMClassifier(nn.Module):
    def __init__(self, num_classes, hidden_dim=416, num_layers=1):
        super(MurilBiLSTMClassifier, self).__init__()
        self.muril = AutoModel.from_pretrained(MODEL_NAME)

        for param in self.muril.parameters():
            param.requires_grad = True

        embedding_dim = 768

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(p=0.1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.muril(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state

        # lstm_out shape: [Batch, Seq_Len, hidden_dim * 2]
        lstm_out, _ = self.bilstm(sequence_output)

        # Replicates return_sequences=True via Global Max Pooling over sequence length
        pooled_out, _ = torch.max(lstm_out, dim=1)

        return self.fc(self.dropout(pooled_out))

# ==========================================
# 4. Training Engine Function
# ==========================================
def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(data_loader)

# ==========================================
# 5. Comprehensive Evaluation Function
# ==========================================
def evaluate_model(model, data_loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))

    return acc, precision, recall, f1, cm

# ==========================================
# 6. Pipeline Execution Block
# ==========================================
# Based on Comparative Study of Marathi Text Classification Using Monolingual and Multilingual Embeddings, I tried this

if __name__ == "__main__":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("--- Reading Local Data Files from Colab Storage ---")
    train_df = pd.read_csv("tweets-train.csv")
    val_df = pd.read_csv("tweets-valid.csv")

    # Force shuffle the rows completely to ensure all classes are blended evenly
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print("Raw Training Labels distribution:\n", train_df['label'].value_counts())
    print("Raw Validation Labels distribution:\n", val_df['label'].value_counts())

    # Defensive dictionary mapping to ensure no class gets systematically dropped
    label_mapping = {
        "negative": 0, "neutral": 1, "positive": 2,
        "0": 0, "1": 1, "2": 2,
        0: 0, 1: 1, 2: 2,
        3: 2, "3": 2
    }

    def clean_labels(df):
        if df['label'].dtype == object:
            df['label'] = df['label'].astype(str).str.strip().str.lower()
        df['label'] = df['label'].map(label_mapping)
        df = df.dropna(subset=['label'])
        return df

    train_df = clean_labels(train_df)
    val_df = clean_labels(val_df)

    train_texts = train_df['tweet'].astype(str).tolist()
    train_labels = train_df['label'].astype(int).tolist()

    val_texts = val_df['tweet'].astype(str).tolist()
    val_labels = val_df['label'].astype(int).tolist()

    print(f"\nFinal Sanitized Class Distribution in Validation Split: {set(val_labels)}")

    # Re-build loaders with securely distributed splits
    train_dataset = MarathiTextDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_dataset = MarathiTextDataset(val_texts, val_labels, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = MurilBiLSTMClassifier(num_classes=NUM_CLASSES).to(DEVICE)

    muril_params = list(model.muril.parameters())
    head_params = list(model.bilstm.parameters()) + list(model.fc.parameters())

    optimizer = torch.optim.Adam([
        {'params': muril_params, 'lr': 1e-6},
        {'params': head_params, 'lr': 1e-4}
    ])
    loss_fn = nn.CrossEntropyLoss()

    print("\n--- Starting Dataset Training Loops ---")
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, train_loader, loss_fn, optimizer, DEVICE)
        acc, prec, rec, f1, cm = evaluate_model(model, val_loader, DEVICE)

        print(f"\n--- Epoch {epoch + 1}/{EPOCHS} ---")
        print(f"Training Loss: {train_loss:.4f}")
        print(f"Validation Metrics -> Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}")
        print("Confusion Matrix:\n", cm)


Execution Target Device: cuda


--- Reading Local Data Files from Colab Storage ---
Raw Training Labels distribution:
 label
 1    4038
-1    4038
 0    4038
Name: count, dtype: int64
Raw Validation Labels distribution:
 label
 1    500
 0    500
-1    500
Name: count, dtype: int64

Final Sanitized Class Distribution in Validation Split: {0, 1}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Starting Dataset Training Loops ---

--- Epoch 1/3 ---
Training Loss: 0.6168
Validation Metrics -> Accuracy: 0.8480 | Precision: 0.8489 | Recall: 0.8480 | F1-Score: 0.8479
Confusion Matrix:
 [[437  63   0]
 [ 89 411   0]
 [  0   0   0]]

--- Epoch 2/3 ---
Training Loss: 0.3672
Validation Metrics -> Accuracy: 0.8580 | Precision: 0.8580 | Recall: 0.8580 | F1-Score: 0.8580
Confusion Matrix:
 [[429  71   0]
 [ 71 429   0]
 [  0   0   0]]

--- Epoch 3/3 ---
Training Loss: 0.3369
Validation Metrics -> Accuracy: 0.8590 | Precision: 0.8598 | Recall: 0.8590 | F1-Score: 0.8589
Confusion Matrix:
 [[418  82   0]
 [ 59 441   0]
 [  0   0   0]]
